In [3]:
import pandas as pd

In [45]:
#Lendo a tabela ab_test_clean
ab_test_clean = pd.read_parquet("../data/ab_test_clean.parquet")
#display(ab_test_clean.head())
#display(ab_test_clean.info())

#Lendo a tabela consumer_clean
consumer_clean = pd.read_parquet("../data/consumer_clean.parquet")
#display(consumer_clean.head())
#display(consumer_clean.info())

#Explorando as tabelas
common_ids = set(ab_test_clean['customer_id']) & set(consumer_clean['customer_id'])
print(f"Número de IDs em comum: {len(common_ids)}")
print(f"Total na tabela ab_test: {ab_test_clean['customer_id'].nunique()}")
print(f"Total na tabela consumer: {consumer_clean['customer_id'].nunique()}")
print(ab_test_clean['is_target'].value_counts(normalize=True))

#Fazendo o merge entre as tabelas mantendo somente os usuários cadastradas na tabela de consumer, por ser mais seguro e limpo
consumer_merge = ab_test_clean.merge(consumer_clean[['customer_id', 'active']], on='customer_id', how='left')
#display(consumer_merge.head())
#display(consumer_merge.info())

#Salvando os dados mergeados entre as tabelas consumer_clean e ab_test_clean
consumer_merge.to_parquet("../data/consumer_merge.parquet", index=False)
#display(consumer_merge.head())
#display(consumer_merge.info())

#Analisando o % de usuários ativos e inativos em cada grupo
perc_ati_ina = consumer_merge.groupby('is_target')['active'].value_counts(normalize=True)
display(perc_ati_ina)

Número de IDs em comum: 806156
Total na tabela ab_test: 806466
Total na tabela consumer: 806156
is_target
target     0.552936
control    0.447064
Name: proportion, dtype: float64


is_target  active
control    True      0.998022
           False     0.001978
target     True      0.998021
           False     0.001979
Name: proportion, dtype: float64

In [ ]:
import plotly.express as px

#Lendo a tabela order_clean
order_clean = pd.read_parquet("../data/order_clean.parquet")
#display(ab_test_clean.head())
#display(ab_test_clean.info())

min_date = order_clean['order_created_at'].min()
max_date = order_clean['order_created_at'].max()
print(min_date, max_date)

common_ids = set(consumer_merge['customer_id']) & set(order_clean['customer_id'])
print(f"Número de IDs em comum: {len(common_ids)}")
print(f"Total na tabela consumer: {consumer_merge['customer_id'].nunique()}")
print(f"Total na tabela order: {order_clean['customer_id'].nunique()}")

order_clean['ano_mes_created_at'] = order_clean['ano_mes_created_at'].astype(str)

#Calculando o número de pedidos e ticket medio por customer_id
order_customer_id = order_clean.groupby(['customer_id', 'ano_mes_created_at'])['order_id'].count().reset_index()
order_customer_id.rename(columns={'order_id': 'qtd_pedidos'}, inplace=True)

ticket_medio_customer_id = order_clean.groupby(['customer_id', 'ano_mes_created_at'])['order_total_amount'].sum().reset_index()

#Fazendo o merge entre as tabelas order_clean e consumer_merge para trazer o is_target
order_customer = consumer_merge.merge(order_customer_id, on='customer_id', how='left')

ticket_merge = consumer_merge.merge(ticket_medio_customer_id, on='customer_id', how='left')

#Preenchendo nulos com zero (usuários sem pedidos)
order_customer['qtd_pedidos'].fillna(0, inplace=True)

ticket_medio_customer_id['order_total_amount'].fillna(0, inplace=True)

#Calculando a média por grupo de consumers
average_order = order_customer.groupby(['is_target', 'ano_mes_created_at'])['qtd_pedidos'].mean()

ticket_medio = ticket_merge.groupby(['is_target', 'ano_mes_created_at'])['order_total_amount'].mean()

display(average_order)
display(ticket_medio)

    #criação do DF
avg_order_df = average_order.reset_index()

ticket_medio_df = ticket_medio.reset_index()

    #criação do gráfico
        #average_order
fig1 = px.line(
    avg_order_df,
    x='ano_mes_created_at',
    y='qtd_pedidos',
    color='is_target',
    markers=True,
    title='Média de Pedidos por Usuário por Mês',
    labels={'ano_mes_created_at': 'Ano-Mês', 'qtd_pedidos': 'Média de Pedidos', 'is_target': 'Grupo'}
)
fig1.update_layout(xaxis_tickangle=-45)
fig1.show()

        #ticket_medio
fig2 = px.line(
    ticket_medio_df,
    x='ano_mes_created_at',
    y='order_total_amount',
    color='is_target',
    markers=True,
    title='Ticket Médio por Usuário por Mês',
    labels={'ano_mes_created_at': 'Ano-Mês', 'order_total_amount': 'Ticket Médio (R$)', 'is_target': 'Grupo'}
)
fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

#Transformando as séries em DataFrames, fazendo o pivot
average_order_df = average_order.reset_index().pivot(index='ano_mes_created_at', columns='is_target', values='qtd_pedidos')

ticket_medio_df = ticket_medio.reset_index().pivot(index='ano_mes_created_at', columns='is_target', values='order_total_amount')

#Calculando a variação de um mês para o outro entre dez/18 e jan/19
average_order_pct = ((average_order_df.loc['2019-01'] - average_order_df.loc['2018-12']) / average_order_df.loc['2018-12']) * 100

ticket_medio_pct = ((ticket_medio_df.loc['2019-01'] - ticket_medio_df.loc['2018-12']) / ticket_medio_df.loc['2018-12']) * 100

display(average_order_pct)
display(ticket_medio_pct)

    #criação do DF
avg_order_pct_df = average_order_pct.reset_index()

ticket_medio_pct_df = ticket_medio_pct.reset_index()

    #criação do gráfico
        #average_order_pct
fig3 = px.bar(
    avg_order_pct_df,
    x='is_target',
    y='qtd_pedidos',
    color='is_target',
    text='qtd_pedidos',
    title='Variação Percentual da Média de Pedidos (Dez/18 → Jan/19)',
    labels={'qtd_pedidos': 'Variação (%)', 'is_target': 'Grupo'}
)
fig3.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig3.update_layout(showlegend=False, yaxis_ticksuffix="%")
fig3.show()

        #ticket_medio_pct
fig4 = px.bar(
    ticket_medio_pct_df,
    x='is_target',
    y='order_total_amount',
    color='is_target',
    text='order_total_amount',
    title='Variação Percentual do Ticket Médio (Dez/18 → Jan/19)',
    labels={'order_total_amount': 'Variação (%)', 'is_target': 'Grupo'}
)
fig4.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig4.update_layout(showlegend=False, yaxis_ticksuffix="%")
fig4.show()


2018-12-03 00:00:00+00:00 2019-01-31 23:59:59+00:00
Número de IDs em comum: 806466
Total na tabela consumer: 806466
Total na tabela order: 806466


/tmp/ipykernel_1030354/2618852042.py:29: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  order_customer['qtd_pedidos'].fillna(0, inplace=True)
/tmp/ipykernel_1030354/2618852042.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace

is_target  ano_mes_created_at
control    2018-12               2.131428
           2019-01               2.803385
target     2018-12               2.300177
           2019-01               3.176947
Name: qtd_pedidos, dtype: float64

is_target  ano_mes_created_at
control    2018-12               102.008186
           2019-01               134.331655
target     2018-12               109.495904
           2019-01               151.886838
Name: order_total_amount, dtype: float64

TypeError: Object of type Period is not JSON serializable